**Necessary Imports**

In [1]:
from bs4 import BeautifulSoup
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from time import sleep
import random
import csv
import time
import pandas as pd

**Random User-Agent for Scraping**

In [2]:
def get_user_agent():
    with open('user-agents.txt') as f:
        ua_strings = f.read().split("\n")
    return random.choice(ua_strings)

**Scraping Links of Those Products**

In [22]:

class HamrobazarScraper:
    def __init__(self, url, driver_path=None, headless=True):
        self.url = url
        ua = get_user_agent()

        self.options = Options()
        self.options.add_argument(f"user-agent={ua}")
        self.options.add_argument("window-size=1400,900")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--disable-dev-shm-usage")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--disable-blink-features=AutomationControlled")
        self.options.add_experimental_option("excludeSwitches", ["enable-logging"])
        if headless:
            # 'new' headless is fine on latest Chrome; fall back to --headless if needed
            self.options.add_argument("--headless=new")

        if driver_path:
            service = Service(driver_path)
        else:
            # If you have webdriver_manager installed:
            # service = Service(ChromeDriverManager().install())
            # Otherwise, rely on PATH:
            service = Service()

        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.wait = WebDriverWait(self.driver, 20)

    def category_name(self):
        """Extract and return category name."""
        self.driver.get(self.url)
        # Wait for the category header to appear
        # If the real DOM uses a different class, adjust here.
        header = self.wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".search--titles"))
        )
        return header.text.replace("Category : ", "").strip()

    
    def _maybe_accept_cookies(self, timeout=5):
            driver = self.driver
            end = time() + timeout
            selectors = [
                "button#onetrust-accept-btn-handler",
                "button[aria-label*='accept']",
                "button:contains('Accept')",  # will be ignored by Selenium but kept for completeness
                "button.accept", "button[mode='accept']",
                "button.cookie-accept",
            ]
            while time() < end:
                for sel in selectors:
                    try:
                        btns = driver.find_elements(By.CSS_SELECTOR, sel)
                        if btns:
                            try:
                                self.wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, sel))).click()
                                return True
                            except Exception:
                                # Try a JS click fallback
                                driver.execute_script("arguments[0].click();", btns[0])
                                return True
                    except Exception:
                        pass
                sleep(0.5)
            return False

    def _switch_into_first_iframe_if_needed(self):
        driver = self.driver
        try:
            iframes = driver.find_elements(By.TAG_NAME, "iframe")
            if iframes:
                driver.switch_to.frame(iframes[0])
                return True
        except Exception:
            pass
        return False

    def _find_cards_any_selector(self):
        driver = self.driver
        # Try a sequence of plausible selectors (update once you confirm the DOM)
        candidates = [
            ".listing-card",
            "div[class*='listing']",
            "article[class*='listing']",
            "li[class*='listing']",
            "[data-testid*='listing']",
        ]
        for css in candidates:
            cards = driver.find_elements(By.CSS_SELECTOR, css)
            if cards:
                return cards, css
        return [], None

    def _extract_card(self, card):
        # Title/link
        link = ""
        name = ""
        try:
            a = card.find_element(By.CSS_SELECTOR, "a[href]")
            link = a.get_attribute("href") or ""
            name = a.text.strip()
        except Exception:
            pass

        # Price – try a few patterns
        price = ""
        for sel in [".price_text", "[class*='price']", "span:contains('रु')", "span:contains('Rs')"]:
            try:
                el = card.find_element(By.CSS_SELECTOR, sel)
                txt = el.text.strip()
                if txt:
                    price = txt
                    break
            except Exception:
                continue

        return name, price, link

    def hamrobazar_automation(self, interval=1.0, max_items=10, max_scrolls=50, initial_wait=12):
        driver = self.driver
        try:
            driver.get(self.url)

            # Handle cookie / consent banners
            self._maybe_accept_cookies()

            # Adaptive wait: scroll until cards appear or timeout
            start = time()
            cards, used_selector = [], None
            last_height = driver.execute_script("return document.documentElement.scrollHeight")

            while time() - start < initial_wait:
                cards, used_selector = self._find_cards_any_selector()
                if cards:
                    break
                driver.execute_script("window.scrollBy(0, Math.floor(window.innerHeight*0.8));")
                sleep(0.7)

            # If still nothing, try switching into the first iframe
            if not cards:
                switched = self._switch_into_first_iframe_if_needed()
                if switched:
                    start2 = time()
                    while time() - start2 < 6:
                        cards, used_selector = self._find_cards_any_selector()
                        if cards:
                            break
                        sleep(0.5)

            if not cards:
                raise TimeoutException("No listing cards found with any known selector (even after scrolling/iframe).")

            all_names, all_prices, all_links = [], [], []
            seen = set()

            scrolls = 0
            while len(all_names) < max_items and scrolls < max_scrolls:
                cards, _ = self._find_cards_any_selector()
                for card in cards:
                    try:
                        name, price, link = self._extract_card(card)
                        if not link or link in seen or not name:
                            continue
                        all_names.append(name)
                        all_prices.append(price)
                        all_links.append(link)
                        seen.add(link)
                        if len(all_names) >= max_items:
                            break
                    except Exception:
                        continue

                if len(all_names) >= max_items:
                    break

                # Incremental scroll
                driver.execute_script("window.scrollBy(0, Math.floor(window.innerHeight*0.9));")
                sleep(interval)

                new_height = driver.execute_script("return document.documentElement.scrollHeight")
                if new_height == last_height:
                    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
                    sleep(interval)
                    new_height = driver.execute_script("return document.documentElement.scrollHeight")
                last_height = new_height
                scrolls += 1

            return all_names[:max_items], all_prices[:max_items], all_links[:max_items]
        finally:
            driver.quit()


**Running Hamrobazar Scrapper**

In [24]:
hamrobazar_url = "https://hamrobazaar.com/category/automobiles/EB9C8147-07C0-4951-A962-381CDB400E37"

start_time = time.time()

scraper = HamrobazarScraper(hamrobazar_url)

category_name = scraper.category_name()
print(f"Scraping Category: {category_name}\n-------------------------------")

results = scraper.hamrobazar_automation(interval=3, max_items=10)

print(results)

end_time = time.time()
print(f"Scraping completed in {round(end_time - start_time, 2)} seconds")

Scraping Category: Automobiles (15095)
-------------------------------


TypeError: 'module' object is not callable. Did you mean: 'time.time(...)'?

**Scraping of a Certain Product**

In [ ]:

class Hamrobazaar:
    def __init__(self, url):
        self.headers = {'User-Agent': get_user_agent()}
        self.url = url
        self.req = requests.get(url, headers=self.headers)

        # Setting up the Selenium driver:
        self.opt = Options()
        self.path = Service('c:\\users\\chromedriver.exe')
        self.selenium_arguments = [f"user-agent= {self.headers}", "window-size=1400,900", '--silent', '--no-sandbox',
                                   'disable-notifications', '--disable-dev-shm-usage', '--disable-gpu']

        # Running the Selenium driver:
        self.opt.add_experimental_option('detach', True)
        self.opt.add_experimental_option('excludeSwitches', ['enable-logging'])

        # Mimicking as a client while making a request to server:
        for arg in self.selenium_arguments:
            self.opt.add_argument(arg)

        self.opt.headless = True
        self.driver = webdriver.Chrome(service=self.path, options=self.opt)
        self.driver.maximize_window()
        self.driver.get(self.url)

    def product_name(self):
        try:
            name = WebDriverWait(self.driver, 10).until(
                (EC.visibility_of_element_located((By.CLASS_NAME, 'title--relative')))).text.strip()
            sleep(2)
            self.driver.quit()
            return name
        except TimeoutException:
            sleep(2)
            self.driver.quit()

    def seller_name(self):
        try:
            s_name = self.driver.find_element(By.CLASS_NAME, 'seller__name--inner').find_element(By.TAG_NAME,
                                                                                                 'a').find_element(
                By.TAG_NAME, 'span').text.strip()
            sleep(2)
            self.driver.quit()

            return s_name
        except TimeoutException:
            sleep(2)
            self.driver.quit()

            return s_name

    def seller_contact(self):
        try:
            contact = self.driver.find_element(By.CLASS_NAME, 'seller__address').find_element(By.TAG_NAME,
                                                                                              'span').text.strip()
            sleep(2)
            self.driver.quit()

            return contact
        except NoSuchElementException:
            contact = "Not available"
            sleep(2)
            self.driver.quit()

            return contact

    def seller_link(self):
        try:
            link = self.driver.find_element(By.CLASS_NAME, 'seller__name--inner').find_element(By.TAG_NAME,
                                                                                               'a').get_attribute(
                'href')
            sleep(2)
            self.driver.quit()

            return link
        except NoSuchElementException:
            link = "Not available"
            sleep(2)
            self.driver.quit()

            return link

    def product_condition(self):
        try:
            condition = self.driver.find_element(By.XPATH,
                                                 '//*[@id="hb__root"]/div/main/div/aside[1]/div/div[1]/label').text.strip()
            sleep(2)
            self.driver.quit()

            return condition
        except NoSuchElementException:
            condition = "N/A"
            sleep(2)
            self.driver.quit()
            return condition